# noise-batch-from-latent — ex2: device-targeted noise with optional unit-sphere normalization

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `noise-batch-from-latent`. Running the final beacon cell reports progress against the `GAN: Noise batch from latent_dim` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Noise batch from latent_dim` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`noise-batch-from-latent`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "noise-batch-from-latent"
DD_SUBTOPIC = "GAN: Noise batch from latent_dim"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Noise batch on a target device + unit-sphere normalization

Ex1 built `(B, L, 1, 1)` standard-normal noise. Two production extensions:

1. **Build directly on a target device** — `device=` kwarg saves a later `.to(device)` (one host→GPU copy, often the slowest step).
2. **Unit-sphere normalization** — divide each per-sample vector by its L2 norm so latent codes lie on the unit hypersphere. Used in StyleGAN, BigGAN, and any setup where you want bounded latent interpolation:

```python
noise = t.randn(B, L, 1, 1, device=device, generator=g)
norms = noise.flatten(1).norm(dim=1)         # (B,)
noise = noise / norms.view(B, 1, 1, 1).clamp_min(1e-8)
```

**Why clamp the norm.** Theoretically the norm of `N(0, I_L)` is never zero, but at low `L` you can hit very small norms numerically. The `clamp_min(1e-8)` makes the divide safe even with `L=2`.

**Sphere prior vs Gaussian prior.** Both train; the sphere prior has smoother interpolations (no probability mass near origin) but tighter support — your G must adapt.

### Exercise 2 — device-targeted noise with optional unit-sphere normalization

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `t.randn(B, L, 1, 1, device=device, generator=g)` followed by optional per-sample L2 normalization (with `clamp_min(1e-8)` safety) to project latent codes onto the unit hypersphere.
> Keywords: dcgan, noise, device, sphere-normalize, L2-norm
> ```

**KCs targeted:** `randn-with-device-kwarg`, `per-sample-l2-normalize-to-unit-sphere`

Implement `ex2_dcgan_noise(batch_size, latent_dim, generator, device, normalize=False)`. Two-mode noise builder:

1. Build standard-normal noise of shape `(batch_size, latent_dim, 1, 1)` directly on `device` (use the `device=` kwarg to `t.randn`, NOT a post-hoc `.to(device)`).
2. Pass the `generator` kwarg through for reproducibility.
3. If `normalize=True`:
   - Compute per-sample L2 norms over the last 3 dims (latent + spatial): `norms = noise.flatten(1).norm(dim=1)` — shape `(B,)`.
   - Divide each sample by its norm, clamping the divisor below by `1e-8` for safety: `noise = noise / norms.view(B, 1, 1, 1).clamp_min(1e-8)`.
4. Return the (possibly normalized) noise tensor.

Important shape contract:
- Output shape ALWAYS `(B, L, 1, 1)`, with or without normalize.
- Output device ALWAYS `device`.
- When `normalize=True`, each sample's L2 norm is `1.0 ± 1e-5`.

Input: `batch_size`, `latent_dim` — ints; `generator` — `t.Generator`; `device` — `t.device` or device-string; `normalize` — bool, default False.
Output: `(B, L, 1, 1)` float32 tensor on `device`.

The visualization plots per-sample L2 norm histograms for both modes side by side — the unnormalized form is `χ`-distributed around √latent_dim, the normalized form is a delta at 1.0.

In [ ]:
def ex2_dcgan_noise(batch_size: int, latent_dim: int, generator: 'torch.Generator',
                    device, normalize: bool = False) -> Tensor:
    """(B, L, 1, 1) noise on `device`. Optional per-sample unit-sphere normalize."""
    raise NotImplementedError()


def _test_ex2():
    # Shape + device — unnormalized.
    cpu = t.device('cpu')
    rng = t.Generator().manual_seed(0)
    noise = ex2_dcgan_noise(8, 100, rng, cpu, normalize=False)
    assert noise.shape == (8, 100, 1, 1), f'shape wrong: {tuple(noise.shape)}'
    assert noise.device == cpu, f'device wrong: {noise.device}'
    assert noise.dtype == t.float32

    # Unnormalized — std should be ~1 (standard normal).
    rng2 = t.Generator().manual_seed(0)
    big = ex2_dcgan_noise(1000, 100, rng2, cpu, normalize=False)
    assert abs(big.std().item() - 1.0) < 0.05, f'unnormalized std should be ~1, got {big.std().item():.4f}'
    assert abs(big.mean().item()) < 0.05, f'unnormalized mean should be ~0, got {big.mean().item():.4f}'

    # Normalized — every sample's L2 norm exactly 1.
    rng3 = t.Generator().manual_seed(0)
    norm_noise = ex2_dcgan_noise(32, 64, rng3, cpu, normalize=True)
    assert norm_noise.shape == (32, 64, 1, 1), 'shape must be preserved by normalize'
    per_sample_norms = norm_noise.flatten(1).norm(dim=1)
    assert per_sample_norms.shape == (32,)
    assert t.allclose(per_sample_norms, t.ones(32), atol=1e-5), (
        f'per-sample norms must all be 1, got mean={per_sample_norms.mean().item():.6f}, '
        f'std={per_sample_norms.std().item():.6f}'
    )

    # Reproducibility — same seed gives same noise (both modes).
    for mode in [False, True]:
        rng_a = t.Generator().manual_seed(42)
        rng_b = t.Generator().manual_seed(42)
        a = ex2_dcgan_noise(4, 8, rng_a, cpu, normalize=mode)
        b = ex2_dcgan_noise(4, 8, rng_b, cpu, normalize=mode)
        assert t.equal(a, b), f'normalize={mode}: same seed must give same noise'

    # Direction preserved — normalizing should not change the unit-vector direction.
    rng4 = t.Generator().manual_seed(7)
    raw = ex2_dcgan_noise(4, 32, rng4, cpu, normalize=False)
    rng4b = t.Generator().manual_seed(7)
    normd = ex2_dcgan_noise(4, 32, rng4b, cpu, normalize=True)
    for i in range(4):
        raw_dir = raw[i].flatten() / raw[i].flatten().norm()
        norm_dir = normd[i].flatten() / normd[i].flatten().norm()
        assert t.allclose(raw_dir, norm_dir, atol=1e-5), f'sample {i}: direction changed by normalize'

    # Safety: clamp_min in divisor — verify no inf/nan even when very low L makes some norms tiny.
    # (Construct a degenerate case by post-zeroing one sample.)
    rng5 = t.Generator().manual_seed(0)
    test = ex2_dcgan_noise(2, 4, rng5, cpu, normalize=False)
    # can't easily force a zero sample through randn; trust the clamp by checking small-L behavior.
    rng6 = t.Generator().manual_seed(0)
    tiny = ex2_dcgan_noise(64, 2, rng6, cpu, normalize=True)
    assert t.isfinite(tiny).all(), 'normalize with small L must stay finite (clamp_min safety)'

    # Input shape passes through a DCGAN first layer.
    import torch.nn as nn
    first = nn.ConvTranspose2d(100, 512, 4, stride=1, padding=0, bias=False)
    rng7 = t.Generator().manual_seed(0)
    n = ex2_dcgan_noise(2, 100, rng7, cpu, normalize=True)
    out = first(n)
    assert out.shape == (2, 512, 4, 4), f'first-layer output wrong: {tuple(out.shape)}'

    # --- Visualization: per-sample L2 norm histogram, unnormalized vs normalized ---
    rng_v = t.Generator().manual_seed(0)
    raw_viz = ex2_dcgan_noise(2000, 100, rng_v, cpu, normalize=False)
    rng_v2 = t.Generator().manual_seed(0)
    norm_viz = ex2_dcgan_noise(2000, 100, rng_v2, cpu, normalize=True)
    raw_norms = raw_viz.flatten(1).norm(dim=1).numpy()
    norm_norms = norm_viz.flatten(1).norm(dim=1).numpy()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.hist(raw_norms, bins=50, color='steelblue', edgecolor='black')
    ax1.axvline(100 ** 0.5, color='red', ls='--', label=f'expected √{100}={100**0.5:.2f}')
    ax1.set_title('unnormalized — χ-like distribution around √L'); ax1.legend()
    ax2.hist(norm_norms, bins=10, range=(0.9, 1.1), color='coral', edgecolor='black')
    ax2.set_xlim(0.9, 1.1)
    ax2.set_title('normalized — delta at 1.0 (unit sphere)')
    for ax in (ax1, ax2):
        ax.set_xlabel('per-sample L2 norm'); ax.set_ylabel('count')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_dcgan_noise(batch_size: int, latent_dim: int, generator: 'torch.Generator',
                    device, normalize: bool = False) -> Tensor:
    noise = t.randn(batch_size, latent_dim, 1, 1, device=device, generator=generator)
    if normalize:
        norms = noise.flatten(1).norm(dim=1)
        noise = noise / norms.view(batch_size, 1, 1, 1).clamp_min(1e-8)
    return noise
```

**`device=` in `t.randn` avoids a host→device copy.** Building on CPU then `.to('cuda')` does two allocations and one PCIe copy. Passing `device=` builds the tensor on the target device directly — a meaningful speedup when sampling thousands of batches in a real training loop.

**Why per-sample (not whole-batch) normalization.** Each sample is an independent latent code; the unit sphere lives in `latent_dim`-space. Normalizing the entire batch tensor as one vector would couple samples together — nonsense.

**`flatten(1)` then `norm(dim=1)`.** Collapses the L + spatial dims into a single feature axis per sample, then computes L2 along it. Returns shape `(B,)`. The `view(B, 1, 1, 1)` broadcasts the per-sample scalar back across the original shape.

**Why `clamp_min(1e-8)`, not `clamp(min=1e-8)`.** Same call — `clamp_min` is the one-arg form. Either works; `clamp_min` reads tighter.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()